In [20]:
import os
import pickle
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
from torch_geometric.data import Data
from torch_geometric.utils import to_networkx
from tqdm import trange

In [2]:
class BaseModel(nn.Module):
    def forward(self, data):
        raise NotImplementedError("Forward method must be implemented.")

# Function to dynamically import model classes
def import_model_class(module_name, class_name):
    module = __import__(module_name, fromlist=[class_name])
    return getattr(module, class_name)

In [3]:
model_classes = [
    ("ral_model", "RAL", {"input_dim": 64, "output_dim": 16}), 
    ("gcn_baseline_model", "GCNBaseline", {"input_dim": 64, "hidden_dim": 16,"output_dim": 1})
]

In [4]:
models = []
for module_name, class_name, params in model_classes:
    model_class = import_model_class(module_name, class_name)
    model_instance = model_class(**params)
    models.append(model_instance)

# Load models or initialize if not present
for model in models:
    model_name = model.__class__.__name__.lower()
    if os.path.exists(f"{model_name}.pth"):
        model.load_state_dict(torch.load(f"{model_name}.pth"))

criterion = nn.MSELoss()

C:\Users\arpit\AppData\Local\Temp\ipykernel_2524\3141508132.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"{model_name}.pth"))


In [5]:

# Training loop
if not all(os.path.exists(f"{model.__class__.__name__.lower()}.pth") for model in models):
    print("Training ...")
    with open("train.pkl", "rb") as f:
        graphs = pickle.load(f)
    train_loader = DataLoader(graphs, batch_size=32, shuffle=True)
    
    optimizers = [torch.optim.Adam(model.parameters(), lr=0.01) for model in models]
    num_epochs = 10
    loop = trange(num_epochs, desc="Training")
    
    for epoch in loop:
        losses = []
        for data in train_loader:
            for model, optimizer in zip(models, optimizers):
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, data.y)
                losses.append(loss.item())
                loss.backward()
                optimizer.step()
        
        loop.set_postfix({"MSE": np.mean(losses)})

    # Save models
    for model in models:
        torch.save(model.state_dict(), f"{model.__class__.__name__.lower()}.pth")

In [26]:
def visualize_graph(G, actual_y, predicted_y):
    # G = nx.DiGraph()
    # # Filter nodes and edges for the given graph_id
    # nodes = node_df[node_df['graph_id'] == graph_id]
    # edges = edge_df[edge_df['graph_id'] == graph_id]
    
    # # Add nodes and edges to the graph
    # G.add_nodes_from(nodes['node_id'].tolist())
    # for _, edge in edges.iterrows():
    #     G.add_edge(edge['source_node'], edge['target_node'], weight=edge['weight'])
    
    # Add layer attribute to nodes based on topological generations
    layers = {node: i for i, layer in enumerate(nx.topological_generations(G)) for node in layer}
    nx.set_node_attributes(G, layers, 'layer')

    abs_differences = np.abs(np.array(predicted_y) - np.array(actual_y))

    labels = {
        node: f"{round(diff, 2)}"
        for node, actual, pred, diff in zip(G.nodes, actual_y, predicted_y, abs_differences)
    }
    print(labels)

    # Draw the graph with multipartite layout
    pos = nx.multipartite_layout(G, subset_key='layer')  # Use multipartite layout for visualization
    plt.figure(figsize=(10, 8))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=500, font_size=10, font_weight='bold', labels=labels)
    print("drawn graph")
    # Display edge weights
    edge_weights = nx.get_edge_attributes(G, 'weight')
    edge_labels = {edge: round(weight, 2) for edge, weight in edge_weights.items()}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)  # Draw edge labels
    plt.title("Graph Visualization with Prediction Differences")
    print("SHOWING GRAPH")
    plt.show()

In [9]:
with open("test.pkl", "rb") as f:
    graphs = pickle.load(f)

test_loader = DataLoader(graphs, batch_size=1000, shuffle=False)

for model in models:
    model.eval()
    with torch.no_grad():
        test_losses = []
        for data in test_loader:
            output = model(data)
            loss = criterion(output, data.y)
            test_losses.append(loss.item())
        print(f"{model.__class__.__name__} Test Loss: {np.mean(test_losses)}")




RAL Test Loss: 0.006100930273532867
GCNBaseline Test Loss: 0.0904838889837265


In [27]:
test_iter = iter(test_loader)
first_graph = next(test_iter)  # Get the first test graph

# Convert the graph to a NetworkX graph for visualization
G = to_networkx(first_graph, to_undirected=False)

# Get actual y values from the first graph
actual_y = first_graph.y.cpu().numpy()

# Get predictions from one of your models
model = models[0]  # Use the first model for example
model.eval()
with torch.no_grad():
    predicted_y = model(first_graph).cpu().numpy()

# Visualize the graph with differences
print("Calling VizGraph")
visualize_graph(G, actual_y, predicted_y)

Calling VizGraph
{0: 'y: 0.8199999928474426, pred: 0.8199999928474426, diff: 0.0', 1: 'y: 0.9599999785423279, pred: 0.9599999785423279, diff: 0.0', 2: 'y: 0.28999999165534973, pred: 0.28999999165534973, diff: 0.0', 3: 'y: 0.7300000190734863, pred: 0.7300000190734863, diff: 0.0', 4: 'y: 0.44999998807907104, pred: 0.4300000071525574, diff: 0.019999999552965164', 5: 'y: 1.159999966621399, pred: 1.1799999475479126, diff: 0.019999999552965164', 6: 'y: 0.5, pred: 0.5099999904632568, diff: 0.0', 7: 'y: 0.4300000071525574, pred: 0.4300000071525574, diff: 0.0', 8: 'y: 0.27000001072883606, pred: 0.27000001072883606, diff: 0.0', 9: 'y: 0.4099999964237213, pred: 0.3199999928474426, diff: 0.09000000357627869', 10: 'y: 0.8100000023841858, pred: 0.8100000023841858, diff: 0.0', 11: 'y: 0.3199999928474426, pred: 0.33000001311302185, diff: 0.0', 12: 'y: 0.9800000190734863, pred: 0.7799999713897705, diff: 0.20000000298023224', 13: 'y: 0.28999999165534973, pred: 0.28999999165534973, diff: 0.0', 14: 'y: 0.